3 Tank Recycle Scenario

In [ ]:
original_dir = "/Users/anelycke/Desktop/Master/"
cd(original_dir)
include("3 Tank Bioreactor.jl")
include("MPC Model 3tk Bioreactor.jl")

assign_weights (generic function with 1 method)

In [3]:
##############################
# ASSIGN WEIGHTS 
##############################
#alpha_tk = assign_weights()
alpha_tk = nothing
##############################
# CREATE MPC MODEL
##############################
mpc_model = create_mpc_model(alpha_tk)

println(alpha_tk)

nothing


In [ ]:
##############################
# RUN MPC
##############################
using JuMP

n_steps = Int(nt/dt)+1     

Vprofile = zeros(n_steps,NI)  
Ssprofile = zeros(n_steps, 1)
Xbprofile = zeros(n_steps, 1)
Soprofile = zeros(n_steps, 1)

F_profile =  zeros(n_steps-1, O)

F_profile[1, :] = f0
Vprofile[1, :] = V0
Ssprofile[1] = Ss0
Xbprofile[1] = Xb0
Soprofile[1] = So0


for i = 2:n_steps

    fmax = fmax_vec[:, i]
    Bd = Bd_vec[:, i]

    # FIX INITIAL CONDITIONS IN MODEL AND RUN 
    fix.(mpc_model[:V][:, 1], V0, force=true)
    fix.(mpc_model[:Ss][1, 1], Ss0, force=true)
    fix.(mpc_model[:Xb][1, 1], Xb0, force=true)
    fix.(mpc_model[:So][1, 1], So0, force=true)

    # FIX DISTURBANCES IN MODEL
    fix.(mpc_model[:fmax], fmax, force=true)
    fix.(mpc_model[:Bd], Bd, force=true)

    # SOLVE MODEL 
    optimize!(mpc_model)
    status = JuMP.termination_status(mpc_model)

    if status != MOI.LOCALLY_SOLVED && status != MOI.OPTIMAL
        println("Model did not converge to global or local optima")
    else
        println("Model solved successfully")
    end
    
    println(status)
    

    # SAVE RESULTS FROM 2ND TIME STEP (FIRST TIME STEP IS INITIAL CONDITION)
    V0 = [value(mpc_model[:V][ni, 2]) for ni in 1:NI]
    Ss0 = value(mpc_model[:Ss][1, 2])
    Xb0 = value(mpc_model[:Xb][1, 2])
    So0 = value(mpc_model[:So][1, 2])

    f0 = value.(mpc_model[:f][:, 1])
    
    Vprofile[i, :] = V0 
    Ssprofile[i] = Ss0 
    Xbprofile[i] = Xb0 
    Soprofile[i] = So0 
    F_profile[i-1, :] = f0



end 


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.19, running with linear solver MUMPS 5.8.1.

Number of nonzeros in equality constraint Jacobian...:     2372
Number of nonzeros in inequality constraint Jacobian.:      924
Number of nonzeros in Lagrangian Hessian.............:     1176

Total number of variables............................:     1330
                     variables with only lower bounds:      856
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:      522
Total number of inequality c

Excessive output truncated after 524468 bytes.

LOCALLY_SOLVED
This is Ipopt version 3.14.19, running with linear solver MUMPS 5.8.1.

Number of nonzeros in equality constraint Jacobian...:     2372
Number of nonzeros in inequality constraint Jacobian.:      924
Number of nonzeros in Lagrangian Hessian.............:     1176

Total number of variables............................:     1330
                     variables with only lower bounds:      856
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:      522
Total number of inequality constraints...............:      556
        inequality constraints with only lower bounds:      138
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:      418

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0 -1.0720529e+04 1.08e+03 9.90e+01  -1.0 0.00e+00    -  0.0